# CADENCE — Interactive Notebook GUI
**C2 Anomaly Detection via Ensemble Network Correlation Evidence**

Run each cell top-to-bottom. The full GUI renders in Cell 10 — no separate server needed.

---

In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets pandas numpy scikit-learn scipy statsmodels matplotlib pyarrow shap --quiet
print('✅ Dependencies ready')

In [ ]:
import json, logging, sys, tempfile, threading, time, io
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as W
from IPython.display import display, HTML, clear_output

REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analytic_pipeline import BDPConfig, BDPPipeline
from analytic_pipeline.generate_synthetic_data import SyntheticDataGenerator, evaluate_detection
print('✅ Imports OK')

In [ ]:
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@400;600&display=swap');
.cad-hdr  { font-family:'IBM Plex Mono',monospace; color:#58a6ff; font-size:1.3rem; font-weight:700; }
.cad-sec  { font-family:'IBM Plex Mono',monospace; font-size:0.7rem; color:#8b949e; text-transform:uppercase; letter-spacing:1.5px; border-bottom:1px solid #30363d; padding-bottom:4px; margin:14px 0 10px; }
.cad-card { background:#161b22; border:1px solid #21262d; border-radius:8px; padding:14px 16px; margin:8px 0; }
.cad-met  { display:inline-block; background:#161b22; border:1px solid #21262d; border-radius:6px; padding:8px 14px; margin:4px; min-width:100px; text-align:center; }
.cad-met .val { font-family:'IBM Plex Mono',monospace; font-size:1.2rem; color:#58a6ff; font-weight:700; }
.cad-met .lbl { font-family:'IBM Plex Sans',sans-serif; font-size:0.7rem; color:#8b949e; }
.cad-log  { font-family:'IBM Plex Mono',monospace; font-size:0.75rem; background:#010409; color:#58a6ff; border:1px solid #21262d; border-radius:6px; padding:10px 12px; max-height:220px; overflow-y:auto; white-space:pre-wrap; line-height:1.55; }
.cad-tbl  { font-family:'IBM Plex Mono',monospace; font-size:0.78rem; border-collapse:collapse; width:100%; }
.cad-tbl th { background:#161b22; color:#8b949e; padding:7px 10px; text-align:left; border-bottom:1px solid #21262d; font-size:0.7rem; }
.cad-tbl td { padding:7px 10px; border-bottom:1px solid #21262d; color:#c9d1d9; }
.cad-tbl tr:nth-child(even) td { background:#0f1318; }
.sc-hi { color:#f85149; font-weight:700; } .sc-md { color:#d29922; font-weight:700; } .sc-lo { color:#3fb950; font-weight:700; }
.dst-ip { color:#f85149; } .per-v { color:#58a6ff; } .mit-v { color:#484f58; font-size:0.7rem; }
.gt-ok { color:#3fb950; } .gt-ms { color:#f85149; } .gt-dc { color:#8b949e; }
.shap-row { display:flex; align-items:center; gap:8px; margin-bottom:5px; }
.shap-f { font-family:'IBM Plex Mono',monospace; font-size:0.72rem; color:#8b949e; width:170px; text-align:right; flex-shrink:0; }
.shap-t { flex:1; height:12px; background:#0d1117; border-radius:3px; overflow:hidden; }
.shap-b { height:100%; background:linear-gradient(90deg,#1f6feb,#58a6ff); border-radius:3px; }
.shap-v { font-family:'IBM Plex Mono',monospace; font-size:0.72rem; color:#58a6ff; width:36px; text-align:right; }
</style>
"""))
display(HTML('<div class="cad-hdr">CADENCE</div><div style="font-family:IBM Plex Mono,monospace;font-size:0.65rem;color:#484f58;letter-spacing:2px">C2 ANOMALY DETECTION VIA ENSEMBLE NETWORK CORRELATION EVIDENCE</div><hr style="border-color:#21262d;margin:8px 0">widespread'))
print('Styling loaded')

In [ ]:
# Pipeline state shared between UI thread and background thread
_st = dict(running=False, logs='', complete=False, artifacts=None, syn_eval=None, labels=None, conn=None)
_lk = threading.Lock()

class _WLH(logging.Handler):
    def emit(self, record):
        with _lk:
            _st['logs'] += self.format(record) + '\n'

_h = _WLH()
_h.setFormatter(logging.Formatter('%(asctime)s  %(levelname)-8s  %(name)s  %(message)s', '%H:%M:%S'))
logging.getLogger().addHandler(_h)
logging.getLogger().setLevel(logging.INFO)
print('Logging configured')

In [ ]:
S = {'description_width': '220px'}
L = W.Layout(width='480px')
H = W.Layout(width='240px')

def sec(t): return W.HTML(f'<div class="cad-sec">{t}</div>')

# Input mode
w_mode  = W.RadioButtons(options=['Synthetic data','Separate files','Unified file'], value='Synthetic data', description='Input mode:', style=S, layout=W.Layout(width='500px'))

# Synthetic params
w_days  = W.IntSlider(value=30, min=3, max=90, step=1, description='Days:', style=S, layout=L)
w_bgr   = W.IntSlider(value=10000, min=1000, max=50000, step=1000, description='BG rows/day:', style=S, layout=L)
w_noisy = W.IntSlider(value=500, min=100, max=5000, step=100, description='Noisy rows/day:', style=S, layout=L)
w_seed  = W.IntText(value=42, description='Seed:', style=S, layout=H)
syn_box = W.VBox([sec('Synthetic Data'), w_days, w_bgr, w_noisy, w_seed])

# File uploads
w_conn = W.FileUpload(description='Conn log:', accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_dns  = W.FileUpload(description='DNS log:',  accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_http = W.FileUpload(description='HTTP log:', accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_ssl  = W.FileUpload(description='SSL log:',  accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_uni  = W.FileUpload(description='Unified:',  accept='.csv,.parquet,.pq', multiple=False, layout=L)
sep_box = W.VBox([sec('Separate Files'), w_conn, w_dns, w_http, w_ssl])
uni_box = W.VBox([sec('Unified File (log_type column)'), w_uni])

# IForest
w_n_est   = W.IntSlider(value=200, min=10, max=2000, step=50, description='n_estimators:', style=S, layout=L)
w_samp    = W.IntSlider(value=3000, min=100, max=50000, step=500, description='max_samples:', style=S, layout=L)
w_contam  = W.FloatSlider(value=0.05, min=0.001, max=0.20, step=0.005, readout_format='.3f', description='contamination:', style=S, layout=L)
w_rs      = W.IntText(value=42, description='random_state:', style=S, layout=H)
w_stab    = W.FloatSlider(value=0.80, min=0.5, max=1.0, step=0.05, description='stability_thresh:', style=S, layout=L)

# Periodicity
w_nlags   = W.IntSlider(value=40, min=5, max=200, step=1, description='acf_nlags:', style=S, layout=L)
w_conf    = W.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.05, description='confidence_thresh:', style=S, layout=L)
w_cv      = W.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.05, description='cv_threshold:', style=S, layout=L)
w_minper  = W.IntSlider(value=60, min=1, max=3600, step=1, description='min_period_s:', style=S, layout=L)

# Corroboration
w_minscore = W.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.01, readout_format='.2f', description='min_score:', style=S, layout=L)
w_dgaent   = W.FloatSlider(value=3.5, min=1.0, max=6.0, step=0.1, description='dga_entropy_thresh:', style=S, layout=L)
w_rarua    = W.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='rare_ua_threshold:', style=S, layout=L)
w_pertol   = W.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.01, description='period_tolerance:', style=S, layout=L)

# Prefilter
w_fanin  = W.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05, description='fanin_threshold:', style=S, layout=L)
w_failed = W.FloatSlider(value=0.90, min=0.0, max=1.0, step=0.05, description='failed_conn_thresh:', style=S, layout=L)

# Pair
w_minflows = W.IntSlider(value=8, min=2, max=100, step=1, description='min_pair_flows:', style=S, layout=L)
w_maxpairs = W.IntSlider(value=5000, min=100, max=50000, step=100, description='max_pairs:', style=S, layout=L)

# Report
w_report = W.Checkbox(value=True, description='Generate HTML report', style=S)

print('Widgets created')

In [ ]:
def build_cfg():
    cfg = BDPConfig()
    cfg.isolation.n_estimators        = w_n_est.value
    cfg.isolation.max_samples         = w_samp.value
    cfg.isolation.contamination       = w_contam.value
    cfg.isolation.random_state        = w_rs.value
    cfg.isolation.stability_threshold = w_stab.value
    cfg.periodicity.acf_nlags             = w_nlags.value
    cfg.periodicity.confidence_threshold  = w_conf.value
    cfg.periodicity.cv_threshold          = w_cv.value
    cfg.periodicity.min_period_s          = float(w_minper.value)
    cfg.corroboration.min_score              = w_minscore.value
    cfg.corroboration.dga_entropy_threshold  = w_dgaent.value
    cfg.corroboration.rare_ua_threshold      = w_rarua.value
    cfg.corroboration.period_tolerance_pct   = w_pertol.value
    cfg.prefilter.dst_fanin_threshold   = w_fanin.value
    cfg.prefilter.failed_conn_threshold = w_failed.value
    cfg.pair.min_pair_flows = w_minflows.value
    cfg.pair.max_pairs      = w_maxpairs.value
    return cfg

def _save_upload(uw):
    if not uw.value: return None
    up = list(uw.value.values())[0]
    suffix = Path(up['metadata']['name']).suffix or '.csv'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.write(up['content']); tmp.flush(); tmp.close()
    return tmp.name

def run_synthetic():
    with _lk: _st.update(running=True, logs='', complete=False, artifacts=None, syn_eval=None)
    try:
        with _lk: _st['logs'] += f'Generating {w_days.value} days (seed={w_seed.value})...\n'
        gen = SyntheticDataGenerator(seed=w_seed.value)
        conn, dns, http, ssl, labels = gen.generate(days=w_days.value, bg_rows_per_day=w_bgr.value, noisy_rows_per_day=w_noisy.value)
        with _lk:
            _st['logs'] += f'conn:{len(conn):,}  dns:{len(dns):,}  http:{len(http):,}  ssl:{len(ssl):,}\n'
            _st['conn'] = conn; _st['labels'] = labels
        cfg = build_cfg()
        cfg.io.query_start = str(pd.to_datetime(conn['timestamp'].min(), unit='s', utc=True))[:19]
        cfg.io.query_end   = str(pd.to_datetime(conn['timestamp'].max(), unit='s', utc=True))[:19]
        tmpdir = tempfile.mkdtemp(prefix='cad_syn_')
        cfg.io.output_dir = Path(tmpdir)/'output'; cfg.io.output_dir.mkdir(parents=True, exist_ok=True)
        cfg._unified_slices = {'conn':conn,'dns':dns,'http':http,'ssl':ssl}
        cp = Path(tmpdir)/'conn.csv'; conn.to_csv(cp, index=False); cfg.io.input_csv = cp
        with _lk: _st['logs'] += 'Running pipeline...\n'
        pipe = BDPPipeline(cfg)
        if w_report.value:
            from analytic_pipeline.report import ReportContext
            rd = cfg.io.output_dir/'report'; rd.mkdir(exist_ok=True)
            with ReportContext(output_dir=rd, open_browser=False) as rpt:
                art = pipe.run(visualize=False); rpt.finalise(art)
        else:
            art = pipe.run(visualize=False)
        syn_eval = None
        if not art.corroboration.empty:
            syn_eval = evaluate_detection(art.corroboration, labels, conn)
            p,r,f = float(syn_eval['precision'].iloc[0]),float(syn_eval['recall'].iloc[0]),float(syn_eval['f1'].iloc[0])
            with _lk: _st['logs'] += f'P={p:.3f}  R={r:.3f}  F1={f:.3f}\n'
        with _lk: _st.update(artifacts=art, syn_eval=syn_eval, complete=True); _st['logs'] += 'Pipeline complete.\n'
    except Exception as e:
        import traceback
        with _lk: _st['logs'] += f'ERROR: {e}\n{traceback.format_exc()}\n'; _st['complete']=True
    finally:
        with _lk: _st['running'] = False

def run_files():
    with _lk: _st.update(running=True, logs='', complete=False, artifacts=None, syn_eval=None)
    try:
        cfg = build_cfg()
        tmpdir = tempfile.mkdtemp(prefix='cad_')
        cfg.io.output_dir = Path(tmpdir)/'output'; cfg.io.output_dir.mkdir(parents=True, exist_ok=True)
        if w_mode.value == 'Unified file':
            up = _save_upload(w_uni)
            if not up: raise ValueError('No unified file uploaded')
            cfg.io.input_unified = Path(up)
            dns_p = http_p = ssl_p = None
        else:
            cp = _save_upload(w_conn)
            if not cp: raise ValueError('No conn log uploaded')
            p = Path(cp)
            if p.suffix.lower() in ('.parquet','.pq'): cfg.io.input_parquet = p
            else: cfg.io.input_csv = p
            dns_p=_save_upload(w_dns); http_p=_save_upload(w_http); ssl_p=_save_upload(w_ssl)
        with _lk: _st['logs'] += 'Running pipeline...\n'
        pipe = BDPPipeline(cfg)
        art = pipe.run(dns_log_path=dns_p, http_log_path=http_p, ssl_log_path=ssl_p, visualize=False)
        with _lk: _st.update(artifacts=art, complete=True); _st['logs'] += 'Pipeline complete.\n'
    except Exception as e:
        import traceback
        with _lk: _st['logs'] += f'ERROR: {e}\n{traceback.format_exc()}\n'; _st['complete']=True
    finally:
        with _lk: _st['running'] = False

print('Pipeline functions ready')

In [ ]:
def render_funnel(art):
    def sl(df): return len(df) if df is not None and not df.empty else 0
    def ss(df,c):
        try: return int(df[c].sum()) if df is not None and not df.empty and c in df.columns else 0
        except: return 0
    ms = [('Raw rows',sl(art.raw)),('Pre-filtered',sl(art.prefiltered)),('IForest anom.',sl(art.anomalies)),
          ("Pairs eval'd",sl(art.sax)),('SAX pass',ss(art.sax,'sax_prescreen_pass')),
          ('Beacon pairs',ss(art.periodicity,'is_beacon_pair')),('Corroborated',ss(art.corroboration,'corroborated'))]
    cards=''.join(f'<div class="cad-met"><div class="val">{v:,}</div><div class="lbl">{l}</div></div>' for l,v in ms)
    display(HTML(f'<div class="cad-sec">Pipeline Funnel</div><div>{cards}</div>'))

def render_shap(art):
    if art.shap_values is None or art.shap_values.empty: return
    sc=[c for c in art.shap_values.columns if c.startswith('shap_') and c!='shap_sum']
    if not sc: return
    ma=art.shap_values[sc].abs().mean().sort_values(ascending=False)
    ma.index=[c.replace('shap_','') for c in ma.index]
    mx=float(ma.max()) or 1.0
    rows=''.join(f'<div class="shap-row"><div class="shap-f">{f}</div><div class="shap-t"><div class="shap-b" style="width:{v/mx*100:.1f}%"></div></div><div class="shap-v">{v:.3f}</div></div>' for f,v in ma.items())
    display(HTML(f'<div class="cad-sec">SHAP — IForest Feature Importance</div><div class="cad-card" style="font-size:0.72rem;color:#8b949e;margin-bottom:8px">Explains why each channel was flagged by Isolation Forest, not why it is C2.</div>{rows}'))

def render_leads(art):
    if art.corroboration is None or art.corroboration.empty:
        display(HTML('<div class="cad-card" style="color:#8b949e">No corroborated leads. Try lowering min_score.</div>')); return
    COLS=['src_ip','dst_ip','dst_port','proto','triage_score','beacon_confidence','dominant_period_s','mitre_techniques']
    show=[c for c in COLS if c in art.corroboration.columns]
    df=art.corroboration[show].sort_values('triage_score',ascending=False).reset_index(drop=True)
    def sc(v):
        try: f=float(v); return 'sc-hi' if f>0.85 else ('sc-md' if f>0.7 else 'sc-lo')
        except: return ''
    hdr=''.join(f'<th>{c}</th>' for c in show)
    rows=''
    for _,row in df.iterrows():
        cells=''
        for col in show:
            v=str(row[col]) if row[col] is not None else ''
            if col=='triage_score': cells+=f'<td class="{sc(v)}">{v}</td>'
            elif col=='dst_ip': cells+=f'<td class="dst-ip">{v}</td>'
            elif col=='dominant_period_s': cells+=f'<td class="per-v">{v}</td>'
            elif col=='mitre_techniques': cells+=f'<td class="mit-v">{v}</td>'
            else: cells+=f'<td>{v}</td>'
        rows+=f'<tr>{cells}</tr>'
    display(HTML(f'<div class="cad-sec">Corroborated Leads</div><div style="overflow-x:auto"><table class="cad-tbl"><thead><tr>{hdr}</tr></thead><tbody>{rows}</tbody></table></div>'))

def render_gt(se):
    if se is None or se.empty: return
    p,r,f=float(se['precision'].iloc[0]),float(se['recall'].iloc[0]),float(se['f1'].iloc[0])
    def mc(lbl,v):
        g=v>=0.75; c='#3fb950' if g else '#f85149'; t='>=0.75' if g else '<0.75'
        return f'<div class="cad-met"><div class="val" style="color:{c}">{v:.3f}</div><div class="lbl">{lbl}</div><div style="font-size:0.65rem;color:{c}">{t}</div></div>'
    dc=[c for c in ['scenario','malicious','detected'] if c in se.columns]
    df=se[dc].copy()
    df['result']=df.apply(lambda r:'Detected' if r.get('detected') else ('MISSED' if r.get('malicious') else 'Correctly suppressed'),axis=1)
    def rc(r): return 'gt-ok' if 'Detected'==r else ('gt-ms' if 'MISSED'==r else 'gt-dc')
    rows=''.join(f'<tr><td>{r["scenario"]}</td><td style="color:{"#d29922" if r.get("malicious") else "#8b949e"}">{"malicious" if r.get("malicious") else "decoy"}</td><td class="{rc(r["result"])}">{r["result"]}</td></tr>' for _,r in df.iterrows())
    display(HTML(f'<div class="cad-sec">Ground-Truth Evaluation</div><div>{mc("Precision",p)}{mc("Recall",r)}{mc("F1",f)}</div><div style="margin-top:10px;overflow-x:auto"><table class="cad-tbl"><thead><tr><th>scenario</th><th>type</th><th>result</th></tr></thead><tbody>{rows}</tbody></table></div>'))

def render_downloads(art):
    display(HTML('<div class="cad-sec">Downloads</div>'))
    for name,df in [('priority',art.priority),('periodicity',art.periodicity),('corroboration',art.corroboration),('shap_values',art.shap_values)]:
        if df is not None and not df.empty:
            out=Path(tempfile.gettempdir())/f'cadence_{name}.csv'; df.to_csv(out,index=False)
            display(HTML(f'<div style="font-family:IBM Plex Mono,monospace;font-size:0.78rem;margin-bottom:4px">cadence_{name}.csv saved to <code>{out}</code> ({len(df):,} rows)</div>'))

print('Rendering functions ready')

In [ ]:
out_log = W.Output()
out_res = W.Output()
out_st  = W.Output()

prog = W.IntProgress(value=0, min=0, max=10, description='Stage 0/10', bar_style='info', layout=W.Layout(width='500px'))
prog.layout.visibility = 'hidden'

run_btn  = W.Button(description='Run Pipeline', button_style='success', icon='play', layout=W.Layout(width='180px', height='38px'))
save_btn = W.Button(description='Save config', layout=W.Layout(width='130px'))
load_btn = W.Button(description='Load config', layout=W.Layout(width='130px'))
cfg_path = W.Text(value='cadence_config.json', description='Config path:', style=S, layout=L)

def _upd_panels(change=None):
    m=w_mode.value
    syn_box.layout.display='flex' if m=='Synthetic data' else 'none'
    sep_box.layout.display='flex' if m=='Separate files' else 'none'
    uni_box.layout.display='flex' if m=='Unified file' else 'none'
w_mode.observe(_upd_panels, names='value'); _upd_panels()

def _save(b):
    import dataclasses
    d=dataclasses.asdict(build_cfg())
    with open(cfg_path.value,'w') as f: json.dump(d,f,indent=2,default=str)
    with out_st: clear_output(); display(HTML(f'<span style="color:#3fb950;font-family:IBM Plex Mono,monospace;font-size:0.8rem">Saved to {cfg_path.value}</span>'))
save_btn.on_click(_save)

def _load(b):
    try:
        with open(cfg_path.value) as f: d=json.load(f)
        iso=d.get('isolation',{})
        if 'n_estimators' in iso: w_n_est.value=iso['n_estimators']
        if 'max_samples' in iso: w_samp.value=iso['max_samples']
        if 'contamination' in iso: w_contam.value=iso['contamination']
        if 'random_state' in iso: w_rs.value=iso['random_state']
        per=d.get('periodicity',{})
        if 'acf_nlags' in per: w_nlags.value=per['acf_nlags']
        if 'confidence_threshold' in per: w_conf.value=per['confidence_threshold']
        if 'min_period_s' in per: w_minper.value=int(per['min_period_s'])
        corr=d.get('corroboration',{})
        if 'min_score' in corr: w_minscore.value=corr['min_score']
        with out_st: clear_output(); display(HTML(f'<span style="color:#3fb950;font-family:IBM Plex Mono,monospace;font-size:0.8rem">Loaded {cfg_path.value}</span>'))
    except Exception as e:
        with out_st: clear_output(); display(HTML(f'<span style="color:#f85149;font-family:IBM Plex Mono,monospace;font-size:0.8rem">{e}</span>'))
load_btn.on_click(_load)

STAGE_KWS=['Stage 1','Stage 2','Stage 3','Stage 4','Stage 5','Stage 6','Stage 7','Stage 8','Stage 9','Stage 10']

def _on_run(b):
    with _lk:
        if _st['running']: return
    run_btn.disabled=True; run_btn.description='Running...'
    prog.value=0; prog.bar_style='info'; prog.layout.visibility='visible'
    with out_res: clear_output()
    with out_log: clear_output()
    t=threading.Thread(target=run_synthetic if w_mode.value=='Synthetic data' else run_files, daemon=True)
    t.start()
    def _poll():
        prev_len=0; start=time.time()
        while True:
            time.sleep(0.8)
            with _lk:
                logs=_st['logs']; done=_st['complete']; art=_st['artifacts']; se=_st['syn_eval']
            if len(logs)!=prev_len:
                prev_len=len(logs)
                with out_log: clear_output(wait=True); display(HTML(f'<div class="cad-log">{logs}</div>'))
            stage=sum(1 for kw in STAGE_KWS if kw+':' in logs or kw+' ' in logs)
            el=int(time.time()-start)
            prog.value=stage; prog.description=f'Stage {stage}/10  {el}s'
            if done: break
        with out_log: clear_output(wait=True); display(HTML(f'<div class="cad-log">{logs}</div>'))
        run_btn.disabled=False; run_btn.description='Run Pipeline'
        if art is not None:
            prog.value=10; prog.bar_style='success'; prog.description=f'Done  {int(time.time()-start)}s'
            with out_res:
                clear_output()
                render_funnel(art); render_shap(art); render_leads(art)
                if se is not None: render_gt(se)
                render_downloads(art)
        else:
            prog.bar_style='danger'
    threading.Thread(target=_poll, daemon=True).start()
run_btn.on_click(_on_run)

cfg_tabs = W.Tab(children=[
    W.VBox([sec('Isolation Forest'), w_n_est, w_samp, w_contam, w_rs, w_stab]),
    W.VBox([sec('Periodicity'),      w_nlags, w_conf, w_cv, w_minper]),
    W.VBox([sec('Corroboration'),    w_minscore, w_dgaent, w_rarua, w_pertol]),
    W.VBox([sec('Prefilter'),        w_fanin, w_failed]),
    W.VBox([sec('Pair'),             w_minflows, w_maxpairs]),
])
for i,t in enumerate(['IForest','Periodicity','Corroboration','Prefilter','Pair']):
    cfg_tabs.set_title(i,t)

gui = W.VBox([
    W.HTML('<div class="cad-sec">Data Input</div>'), w_mode, syn_box, sep_box, uni_box,
    W.HTML('<div class="cad-sec">Configuration</div>'), cfg_tabs,
    W.HTML('<div class="cad-sec">Config I/O</div>'), cfg_path, W.HBox([save_btn, load_btn]), out_st,
    W.HTML('<div class="cad-sec">Run</div>'), w_report, run_btn, prog,
    W.HTML('<div class="cad-sec">Output</div>'), out_log, out_res,
])
display(gui)